In [0]:
from pyspark.sql.functions import col,current_timestamp,input_file_name
import re

In [0]:
display(dbutils.fs.ls("/Volumes/investment_pyspark/bronze/landing/dividend/raw_files/dividends_2024-04-01_to_2025-03-31.csv"))

In [0]:
# Clear stale schema and checkpoint for a fresh start
'''schema_path = "/Volumes/investment_pyspark/bronze/landing/dividend/dividends_schema/"
checkpoint_path = "/Volumes/investment_pyspark/bronze/landing/dividend/dividends_checkpoint/"

dbutils.fs.rm(schema_path, True)
dbutils.fs.rm(checkpoint_path, True)'''


In [0]:
# 1. Source Path (Where incoming CSV files land)
source_path = "/Volumes/investment_pyspark/bronze/landing/dividend/raw_files/"

# 2. Metadata Paths (Placed completely outside raw_files)
schema_path = "/Volumes/investment_pyspark/bronze/landing/dividend/metadata/_schemas/"
checkpoint_path = "/Volumes/investment_pyspark/bronze/landing/dividend/metadata/_checkpoints/"

# Clear stale schema and checkpoint for a fresh start
#dbutils.fs.rm(schema_path, True)
#dbutils.fs.rm(checkpoint_path, True)

# Drop existing table to avoid duplicates on reprocessing
#spark.sql("DROP TABLE IF EXISTS investment_pyspark.bronze.dividends_raw")

def clean_column_names(col_name):
    col_name = col_name.strip()
    col_name = re.sub(r"[^a-zA-Z0-9_]","_",col_name)
    col_name = re.sub(r"_+", "_",col_name)
    return col_name.strip('_')

df_bronze = (spark.readStream.format("cloudFiles")
.option("cloudFiles.format", "csv")
.option("cloudFiles.schemaLocation", schema_path)
.option("pathGlobFilter", "*.csv")
.option("header","true")
#.option("inferSchema","true")
.load(source_path)
)
df_cleaned = df_bronze.toDF(*[clean_column_names(c) for c in df_bronze.columns])
df_transformed = df_cleaned.withColumn(
    "_ingestion_timestamp",current_timestamp()
).withColumn("_source_file",col("_metadata.file_path"))

query= (
    df_transformed.writeStream.format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("investment_pyspark.bronze.dividends_raw")
)
print(f"Stream started. Waiting for termination...")
query.awaitTermination()
print(f"Stream completed. Processed {query.lastProgress}")

In [0]:
%sql
select * from investment_pyspark.bronze.dividends_raw;

In [0]:
%sql
--drop table if exists investment_pyspark.bronze.dividends_raw;
--select * from investment_pyspark.bronze.dividends_raw;